# OS-SE-AMR (IEEE TWC / TCOM) — Google Colab L4 24 GB Top-Journal Suite

**Architecture**: Drive-backed persistence (`--resume`) + local NVMe SSD dataset caching + BF16 AMP (`--amp`).
Even if Colab disconnects mid-run, re-running any batch cell resumes seamlessly from the last completed step.

In [ ]:
# Cell 1: Verify L4 24 GB GPU & Mount Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/OS-SE-AMR'
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/outputs/journal_twc', exist_ok=True)
print('Drive Root ready:', DRIVE_ROOT)

In [ ]:
# Cell 2: Clone / Pull Code from GitHub into Fast Local SSD (/content/OS-SE-AMR)
REPO_URL = 'https://github.com/camellia-AFK/OS-SE-AMR.git'
%cd /content
import os, subprocess, getpass
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
def _auth_url(tok: str) -> str:
    return REPO_URL.replace('https://', f'https://x-access-token:{tok.strip()}@')
token = ''
try:
    from google.colab import userdata
    token = (userdata.get('GITHUB_TOKEN') or '').strip()
except Exception:
    pass
if not os.path.exists('/content/OS-SE-AMR/.git'):
    if token:
        subprocess.run(['git', 'clone', _auth_url(token), 'OS-SE-AMR'], check=True)
        subprocess.run(['git', '-C', 'OS-SE-AMR', 'remote', 'set-url', 'origin', REPO_URL], check=True)
    else:
        res = subprocess.run(['git', 'clone', REPO_URL, 'OS-SE-AMR'], env=env)
        if res.returncode != 0:
            token = getpass.getpass('Repo is private or unreachable. Paste GitHub PAT (hidden): ').strip()
            subprocess.run(['git', 'clone', _auth_url(token), 'OS-SE-AMR'], check=True)
            subprocess.run(['git', '-C', 'OS-SE-AMR', 'remote', 'set-url', 'origin', REPO_URL], check=True)
else:
    res = subprocess.run(['git', '-C', '/content/OS-SE-AMR', 'pull'], env=env)
    if res.returncode != 0:
        token = token or getpass.getpass('Enter GitHub PAT to pull private repo (hidden): ').strip()
        subprocess.run(['git', '-C', '/content/OS-SE-AMR', 'pull', _auth_url(token)], check=True)
%cd /content/OS-SE-AMR
!pip install -q h5py scikit-learn scipy pyyaml matplotlib pytest hdbscan

In [ ]:
# Cell 3: Stage RML2016.10a + Independent Staging/Download of RML2016.10b & Compact RML2018 Cache
import shutil, os
os.makedirs('/content/OS-SE-AMR/data', exist_ok=True)
if os.path.exists(f'{DRIVE_ROOT}/data/RML2016.10a_dict.pkl'):
    shutil.copy(f'{DRIVE_ROOT}/data/RML2016.10a_dict.pkl', '/content/OS-SE-AMR/data/')
if os.path.exists(f'{DRIVE_ROOT}/data/RML2016.10b.dat'):
    shutil.copy(f'{DRIVE_ROOT}/data/RML2016.10b.dat', '/content/OS-SE-AMR/data/')
elif not os.path.exists('/content/OS-SE-AMR/data/RML2016.10b.dat'):
    !python scripts/download_datasets.py --datasets rml2016b --mirror hf --backend curl --output-dir data
    shutil.copy('data/RML2016.10b.dat', f'{DRIVE_ROOT}/data/RML2016.10b.dat')
if os.path.exists(f'{DRIVE_ROOT}/data/rml2018_cache_1000.h5'):
    shutil.copy(f'{DRIVE_ROOT}/data/rml2018_cache_1000.h5', '/content/OS-SE-AMR/data/')
elif not os.path.exists('/content/OS-SE-AMR/data/rml2018_cache_1000.h5'):
    !python scripts/download_datasets.py --datasets rml2018 --mirror hf --backend curl --output-dir data
    !python scripts/prepare_rml2018.py --input data/GOLD_XYZ_OSC.0001_1024.hdf5 --output data/rml2018_cache_1000.h5 --max-per-cell 1000
    shutil.copy('data/rml2018_cache_1000.h5', f'{DRIVE_ROOT}/data/rml2018_cache_1000.h5')
!ls -lh data

In [ ]:
# Cell 4: Run Full Unit Test Suite Before GPU Experiments
!pytest -v

In [ ]:
# Cell 5: Batch 1 (~25-30 min) — Upgrades, 8-Method SOTA, Cross-Backbone, & Zero-Novelty
!python scripts/run_journal_suite.py --config configs/journal_twc.yaml --batch 1 --amp --resume --output-dir {DRIVE_ROOT}/outputs/journal_twc

In [ ]:
# Cell 6: Batch 2 (~35-45 min) — RML2016.10b & RML2018.01a (1024-pt, 5-Round Long Horizon)
!python scripts/run_journal_suite.py --config configs/journal_twc.yaml --batch 2 --amp --resume --output-dir {DRIVE_ROOT}/outputs/journal_twc

In [ ]:
# Cell 7: Batch 3 (~45 min) & Batch 4 (~5 min) — 5-Seed t-Test, Sensitivity, DOCO++, LaTeX & Vector PDFs
!python scripts/run_journal_suite.py --config configs/journal_twc.yaml --batch 3 --amp --resume --output-dir {DRIVE_ROOT}/outputs/journal_twc
!python scripts/run_journal_suite.py --config configs/journal_twc.yaml --batch 4 --amp --resume --output-dir {DRIVE_ROOT}/outputs/journal_twc
!ls -lh {DRIVE_ROOT}/outputs/journal_twc/batch4/figures {DRIVE_ROOT}/outputs/journal_twc/batch4/latex